# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://doi.org/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library. The dataset is described by a Croissant schema and is AI-ready, demonstrating best practices for data infrastructure in Africa.

### Dataset Source
The dataset is described using a Croissant schema available at the following URL:

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

We begin by loading the dataset metadata and an overview of the records and fields using `mlcroissant`. This will provide information about available record sets and their structure for further exploration.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Set the URL to the Croissant schema
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Let's review the available record sets, their `@id`s, and the fields contained within each record set. All entities are referenced by their Croissant `@id` to ensure clarity and reproducibility.

In [ ]:
# List all record sets and fields using their @id
print("Available record sets (by @id and name):\n")
for record_set in metadata.record_sets:
    print(f"@id: {record_set.id}  |  name: {record_set.name}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    @id: {field.id}  |  name: {field.name}  |  dataType: {field.data_type}")
    print()

## 3. Data Extraction

We extract all available record sets using their `@id` (as listed above). You can select which record set(s) to load based on your analytic needs. For demonstration, we extract all into Pandas DataFrames and preview the columns and records in one record set.

In [ ]:
# Gather all record set @ids
record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist() if not df.empty else 'No columns'}")
    if not df.empty:
        print(df.head(2))
    print()

# For further steps, pick the main record set (assumed to be the first for demonstration):
main_record_set_id = record_set_ids[0]
print(f"\nSelected main record set for EDA: {main_record_set_id}")
main_df = dataframes[main_record_set_id]

## 4. Exploratory Data Analysis (EDA)

We perform basic data processing tasks such as:
- Filtering records based on threshold values in numeric columns (e.g. GAD-7, PHQ-9, PSQ scores)
- Normalizing a numeric field
- Grouping by a demographic attribute

**Note:** Use Croissant `@id` for each field/column, as seen above. We'll check for commonly-used numeric and grouping fields and process accordingly.

In [ ]:
# --- Identify a numeric field (e.g., GAD-7 score) and a group field (e.g., gender or age group) by their @id ---

# Inspect the field @ids in the selected (main) record set
record_set = [rs for rs in metadata.record_sets if rs.id == main_record_set_id][0]
numeric_field_id = None
group_field_id = None
for field in record_set.fields:
    # Heuristic: pick a well-known mental health score, or age, for numeric analysis
    if field.name.lower().startswith('gad') or field.name.lower().startswith('phq') or field.name.lower().startswith('psq'):
        numeric_field_id = field.id
        break

# Heuristic group field: look for 'gender' or 'village' or similar
for field in record_set.fields:
    if 'gender' in field.name.lower() or 'sex' in field.name.lower():
        group_field_id = field.id
        break
if not group_field_id:
    # fallback: group by 'village' if exists
    for field in record_set.fields:
        if 'village' in field.name.lower():
            group_field_id = field.id
            break

if numeric_field_id is None:
    raise ValueError("No suitable numeric field (e.g. GAD or PHQ score) found.")
if group_field_id is None:
    raise ValueError("No suitable group field (e.g. gender or village) found.")

# Use @ids as columns (mlcroissant creates columns matching field @ids for traceability)
numeric_column = numeric_field_id
group_column = group_field_id
print(f"Using numeric field @id: {numeric_column}")
print(f"Using group field @id: {group_column}")

# Drop records with missing values in selected fields
main_df_clean = main_df.dropna(subset=[numeric_column, group_column])

# Ensure the numeric field is treated as float
main_df_clean[numeric_column] = pd.to_numeric(main_df_clean[numeric_column], errors='coerce')
main_df_clean = main_df_clean.dropna(subset=[numeric_column])

# Filter for high score (threshold=10 by default)
threshold = 10
filtered_df = main_df_clean[main_df_clean[numeric_column] > threshold]
print(f"Filtered records where {numeric_column} > {threshold} (n={len(filtered_df)}):")
print(filtered_df[[numeric_column, group_column]].head())

# Add normalized score column
filtered_df[f"{numeric_column}_normalized"] = (filtered_df[numeric_column] - filtered_df[numeric_column].mean()) / filtered_df[numeric_column].std()
print(f"\nNormalized {numeric_column} for filtered records:")
print(filtered_df[[numeric_column, f"{numeric_column}_normalized"]].head())

# Group and compute group statistics
grouped_df = filtered_df.groupby(group_column)[numeric_column].agg(['mean', 'count', 'std'])
print(f"\nStatistics by group ({group_column}):")
print(grouped_df.head())

## 5. Visualization

Visualize the distribution of the selected numeric field (e.g., a mental health assessment score) by group (e.g., gender or village).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
sns.boxplot(data=main_df_clean, x=group_column, y=numeric_column)
plt.title(f"Distribution of {numeric_column} by {group_column}")
plt.xlabel(group_column)
plt.ylabel(numeric_column)
plt.show()

# Histogram
plt.figure(figsize=(8,4))
sns.histplot(main_df_clean[numeric_column].dropna(), kde=True, bins=10)
plt.title(f"Histogram of {numeric_column}")
plt.xlabel(numeric_column)
plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to load and analyze the FAIR² Mental Health Survey Dataset using the Croissant schema and the `mlcroissant` library:
- We reviewed the compositional structure of the dataset via Croissant metadata and explored available record sets and fields using their exact `@id`s.
- We extracted data into Pandas DataFrames, selected numeric mental health assessment fields, filtered and normalized scores, and visualized group-wise distributions.
- These steps enable transparent and reproducible workflows, linking every analysis step back to the original schema-designated field IDs for future interoperability.

Feel free to adapt this template for domain-specific analyses (e.g., deep dives into PHQ-9, subgroup stratifications, or predictive modeling).